# Water-Reflection Waveform

Render the waveform extracted from the video. If the data files are missing,
the pipeline is run first (`wave-from-video`). See `spec.md` for the design.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "output"
npz_path = OUT / "waveform.npz"

if not npz_path.exists():
    from wave_from_video.cli import run
    run(ROOT / "data" / "IMG_4767_2.MOV", OUT)

data = np.load(npz_path)
fps = float(data["fps"])
print({k: data[k].shape for k in data.files})
print("fps:", fps, "wav_sample_rate:", int(data["wav_sample_rate"]))

## Overlay validation
The extracted centerline drawn on the representative frame.

In [ ]:
display(Image(filename=str(OUT / "overlay.png")))

## Spatial waveform
Band amplitude across the representative frame (amplitude vs x).

In [ ]:
plt.figure(figsize=(11, 3))
plt.plot(data["spatial_x"], data["spatial_amplitude"], color="tab:blue", lw=1)
plt.axhline(0, color="grey", lw=0.5)
plt.title("Spatial waveform (representative frame)")
plt.xlabel("x (pixels)")
plt.ylabel("amplitude (px)")
plt.show()

## Temporal waveform
How the reflection oscillates over time (one envelope sample per frame).

In [ ]:
env = data["envelope"]
t = np.arange(env.size) / fps
plt.figure(figsize=(11, 3))
plt.plot(t, env, color="tab:red", lw=0.8)
plt.title("Temporal waveform (oscillation over time)")
plt.xlabel("time (s)")
plt.ylabel("envelope (px)")
plt.show()

## Audio
The signals mapped to playable WAVs (normalised, resampled to 44.1 kHz).

In [ ]:
print("spatial.wav")
display(Audio(filename=str(OUT / "spatial.wav")))
print("temporal.wav")
display(Audio(filename=str(OUT / "temporal.wav")))